# Rare-state benchmark — generate (BioEmu) and score

Runs the generation step that needs a GPU, then scores against the frozen, pre-registered CV.

**Setup:** Colab → Runtime → Change runtime type → **GPU (A100 recommended)**, then Run all.

Set `TARGET` in the params cell to `"t4l"` (excited state vs 2LC9, ~3% experimental) or
`"kras"` (switch-II-open recall). Re-run the whole notebook per target.

Output: `outputs/benchmark/<target>_benchmark_bioemu.json` (headline numbers) and
`outputs/benchmark/<target>_hist.png` (the figure).

In [ ]:
# 1) Install BioEmu (+ helpers), then REPAIR the environment.
# bioemu[cuda] drags in protobuf 7, which breaks the TensorFlow that BioEmu's vendored
# AlphaFold must import for sequence embeddings (undefined-symbol in tflite). Fix = install
# one self-consistent CPU TensorFlow, which pulls a TF-compatible protobuf (<6). Keep
# numpy at 2.x (Colab's compiled stack, e.g. mdtraj, is built for numpy 2; do NOT downgrade).
# TF only needs to be importable for embeddings; sampling runs on torch/GPU.
# After this cell: RESTART THE RUNTIME, then run from Cell 2. Do NOT re-run Cell 1.
!pip -q install "bioemu[cuda]" MDAnalysis matplotlib pyyaml
!pip -q uninstall -y tensorflow tensorflow-cpu tensorflow-gpu tensorflow-intel
!pip -q install "tensorflow-cpu==2.18.1"
import numpy; print("numpy:", numpy.__version__, "(should be 2.x)")
print(">>> Runtime menu > Restart session, then run from Cell 2. Do NOT re-run Cell 1. <<<")

In [ ]:
# 2) Repo + ground-truth structures. Re-run-proof: absolute paths, no nested %cd.
import os
os.chdir("/content")
if not os.path.isdir("/content/conformational-ml/.git"):
    !git clone -q https://github.com/realakshayk1/conformational-ml
os.chdir("/content/conformational-ml")
!git pull -q
!python scripts/fetch_ground_truth_structures.py
print("cwd:", os.getcwd())

In [ ]:
# 3) Parameters. Switch TARGET between 't4l' and 'kras'.
TARGET = "t4l"           # "t4l" or "kras"
N_SAMPLES = 1000          # samples per seed
N_SEEDS = 3               # independent runs (BioEmu draws i.i.d.; no seed flag, so separate runs)

CONFIGS = {
    "t4l": (
        "configs/evaluation/t4l_excited_state.yaml",
        "MNIFEMLRIDEGLRLKIYKDTEGYYTIGIGHLLTKSPSLNAAKSELDKAIGRNTNGVITKDEAEKLFNQ"
        "DVDAAVRGILRNAKLKPVYDSLDAVRRAAAINMVFQMGETGVAGFTNSLRMLQQKRWDEAAVNLAKSRW"
        "YNQTPNRAKRVITTFRTGTWDAYK"),
    "kras": (
        "configs/evaluation/kras_sii_state.yaml",
        "TEYKLVVVGADGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETSLLDILDTAGQEEYSAMRDQ"
        "YMRTGEGFLLVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKSDLPSRTVDTKQAQDLARSYGI"
        "PFIETSAKTRQGVDDAFYTLVREIRKHKEK"),
}
CONFIG, SEQ = CONFIGS[TARGET]
GEN_DIR = f"outputs/benchmark/generated/{TARGET}/bioemu"
print(TARGET, len(SEQ), "residues ->", GEN_DIR)

In [ ]:
# 4) Generate N_SEEDS independent runs of N_SAMPLES. Tries the Python API, falls back to the CLI.
import os, glob, subprocess

def generate(out_dir):
    os.makedirs(out_dir, exist_ok=True)
    try:
        from bioemu.sample import main as bioemu_sample
        bioemu_sample(sequence=SEQ, num_samples=N_SAMPLES, output_dir=out_dir)
    except Exception as e:
        print(f"  Python API failed ({e}); falling back to CLI")
        subprocess.run(["python", "-m", "bioemu.sample", "--sequence", SEQ,
                        "--num_samples", str(N_SAMPLES), "--output_dir", out_dir], check=True)

for seed in range(N_SEEDS):
    out = f"{GEN_DIR}/seed{seed}"
    if glob.glob(f"{out}/samples.*"):
        print("skip (exists):", out); continue
    print("generating:", out)
    generate(out)

In [ ]:
# 5) Convert each seed's BioEmu output (topology.pdb + samples.xtc) to one multi-model samples.pdb.
import glob, os, MDAnalysis as mda
for seed_dir in sorted(glob.glob(f"{GEN_DIR}/seed*")):
    tops = sorted(glob.glob(f"{seed_dir}/topology*.pdb")) or [
        p for p in sorted(glob.glob(f"{seed_dir}/*.pdb")) if os.path.basename(p) != "samples.pdb"]
    xtcs = sorted(glob.glob(f"{seed_dir}/*.xtc"))
    if xtcs and tops:
        u = mda.Universe(tops[0], xtcs[0])
    elif tops:
        u = mda.Universe(tops[0])           # already a multi-model PDB
    else:
        raise FileNotFoundError(f"no topology/trajectory in {seed_dir}")
    ca = u.select_atoms("protein")
    with mda.Writer(f"{seed_dir}/samples.pdb", multiframe=True, n_atoms=ca.n_atoms) as w:
        for _ in u.trajectory:
            w.write(ca)
    print(seed_dir, "->", u.trajectory.n_frames, "models")

In [ ]:
# 6) Score with the frozen CV (geometry-gated, seed mean/sd). Writes the summary + per-conformer records.
OUT = f"outputs/benchmark/{TARGET}_benchmark_bioemu.json"
REC = f"outputs/benchmark/{TARGET}_records_bioemu.json"
!python scripts/run_benchmark.py --generated-dir {GEN_DIR} --config {CONFIG} \
    --label bioemu_{TARGET} --out {OUT} --records-out {REC}

In [ ]:
# 7) Figure + headline.
import json, numpy as np, yaml, matplotlib.pyplot as plt
summary = json.load(open(OUT)); records = json.load(open(REC))
cfg = yaml.safe_load(open(CONFIG))
margin = cfg["contrast"]["margin_angstrom"]; exp = cfg.get("experimental_target_population")
deltas = np.array([r["delta"] for r in records]); o = summary["overall"]

plt.figure(figsize=(7, 4))
plt.hist(deltas, bins=40, color="#4C72B0", alpha=0.85)
plt.axvline(margin, color="crimson", ls="--", label=f"target threshold (\u0394 > {margin})")
plt.xlabel("\u0394 = RMSD_to_ground \u2212 RMSD_to_target (\u00c5)   [\u0394 > 0 \u21d2 closer to target state]")
plt.ylabel("conformers")
plt.title(f"{TARGET.upper()} BioEmu  (n={o['n_conformers']})")
plt.legend(); plt.tight_layout()
plt.savefig(f"outputs/benchmark/{TARGET}_hist.png", dpi=150); plt.show()

exp_str = f"(experimental ~{exp*100:.0f}%)" if exp else "(recall; no published population)"
print(f"HEADLINE  target-state-like: {o['frac_target_state_like']*100:.2f}%  {exp_str}")
print(f"          seed mean\u00b1sd: {summary['headline_frac_target_mean']*100:.2f}% "
      f"\u00b1 {(summary['headline_frac_target_sd'] or 0)*100:.2f}%")
print(f"          geometry-valid target: {(o['frac_target_state_like_geom_valid'] or 0)*100:.2f}%")
print(f"          min RMSD to target: {o['min_rmsd_to_target']:.2f} \u00c5")
print(f"          ground-basin recall: {o['ground_basin_recall']*100:.2f}%")
print(f"          geometry validity rate: {o['geometry_validity_rate']*100:.2f}%")